In [3]:
%cd /Users/arikreuter/Documents/PhD/CausalPriorFitting/

/Users/arikreuter/Documents/PhD/CausalPriorFitting


In [4]:
%load_ext autoreload
%autoreload 2
import warnings
import numpy as np
import torch
import random
import sys
import pandas as pd
sys.path.append("..")

warnings.filterwarnings('ignore') # ignore warnings
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


# Set seeds for reproducibility
seed = 82718
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
# Run only baselines (without CausalPFN) to avoid kernel crashes
%autoreload 2
from CausalPFN.benchmarks import IHDPDataset, ACIC2016Dataset
from CausalPFN.benchmarks import RealCauseLalondeCPSDataset, RealCauseLalondePSIDDataset
import time
from CausalPFN.benchmarks.base import CATE_Dataset, ATE_Dataset
from CausalPFN.benchmarks.baselines import (
    TLearnerBaseline,
    SLearnerBaseline,
    XLearnerBaseline,
    BaselineModel
)

from CausalPFN.src.causalpfn.evaluation import calculate_pehe
from tqdm import tqdm

# Get different realizations for each dataset (only the first two realization - you can change `n_tables`)
datasets = {
    "IHDP": IHDPDataset(n_tables=2),
    "ACIC 2016": ACIC2016Dataset(n_tables=2),
    "RealCause Lalonde CPS": RealCauseLalondeCPSDataset(n_tables=2),
    "RealCause Lalonde PSID": RealCauseLalondePSIDDataset(n_tables=2),
}

# get all of the baselines to compare with
baselines = {
    "X-Learner (no HPO)": XLearnerBaseline(hpo=False),
    "S-Learner (no HPO)": SLearnerBaseline(hpo=False),
    "T-Learner (no HPO)": TLearnerBaseline(hpo=False),
}

# Initialize results DataFrame
results_baselines = pd.DataFrame(columns=["dataset", "realization", "method", "ate_rel_err", "cate_pehe", "ate_time", "cate_time"])

# Iterate through datasets and realizations
pbar = tqdm(
    total=sum(len(dataset) * len(baselines) for dataset in datasets.values()),
    desc="Processing baselines only",
)

for dataset_name, dataset in datasets.items():
    for realization_idx in range(len(dataset)):
        res = dataset[realization_idx]
        cate_dset: CATE_Dataset = res[0]
        ate_dset: ATE_Dataset = res[1]
        true_ate = ate_dset.true_ate

        for method_name, baseline in baselines.items():
            pbar.set_postfix({"dataset": dataset_name, "method": method_name})
            baseline: BaselineModel

            # run baseline estimator for ATE
            start_time = time.time()
            ate_pred = baseline.estimate_ate(X=ate_dset.X, t=ate_dset.t, y=ate_dset.y)
            rel_err = np.abs(ate_pred - true_ate) / np.abs(true_ate)
            ate_time = time.time() - start_time

            # run baseline estimator for CATE
            start_time = time.time()
            cate_pred = baseline.estimate_cate(X_train=cate_dset.X_train, t_train=cate_dset.t_train, y_train=cate_dset.y_train, X_test=cate_dset.X_test)
            cate_pehe = calculate_pehe(cate_dset.true_cate, cate_pred)
            cate_time = time.time() - start_time

            # add results for baseline
            row = dict(
                dataset=dataset_name,
                realization=realization_idx,
                method=method_name,
                ate_rel_err=round(rel_err, 2),
                cate_pehe=round(cate_pehe, 2),
                ate_time=round(ate_time / (ate_dset.X.shape[0] + ate_dset.X.shape[0]) * 100, 2),
                cate_time=round(cate_time / (cate_dset.X_train.shape[0] + cate_dset.X_test.shape[0]) * 100, 2),
            )
            pbar.update(1)
            results_baselines = pd.concat([results_baselines, pd.DataFrame([row])], ignore_index=True)

pbar.close()
print("\nBaselines completed successfully!")
print(results_baselines)

Processing baselines only: 100%|██████████| 24/24 [00:09<00:00,  2.56it/s, dataset=RealCause Lalonde PSID, method=T-Learner (no HPO)]


Baselines completed successfully!
                   dataset realization              method  ate_rel_err  \
0                     IHDP           0  X-Learner (no HPO)         0.00   
1                     IHDP           0  S-Learner (no HPO)         0.02   
2                     IHDP           0  T-Learner (no HPO)         0.02   
3                     IHDP           1  X-Learner (no HPO)         0.01   
4                     IHDP           1  S-Learner (no HPO)         0.02   
5                     IHDP           1  T-Learner (no HPO)         0.01   
6                ACIC 2016           0  X-Learner (no HPO)         0.28   
7                ACIC 2016           0  S-Learner (no HPO)         0.58   
8                ACIC 2016           0  T-Learner (no HPO)         0.36   
9                ACIC 2016           1  X-Learner (no HPO)         0.01   
10               ACIC 2016           1  S-Learner (no HPO)         0.06   
11               ACIC 2016           1  T-Learner (no HPO)       